In [3]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

def generate_dummy_data():
    """Generates dummy data for bank job skills.
    
    The function creates a predefined mapping of job titles to lists of skills. 
    It then generates a DataFrame containing random samples of employees, their skills, 
    and associated job titles. Each job will have 20 dummy samples, with varied skills 
    picked from the corresponding skill sets.
    
    Returns:
        DataFrame: A DataFrame containing employee_id, skills, and job title.
    """
    bank_jobs_skills = {
        "Data Analyst": [["Python", "Data Analysis"], ["Excel", "SQL"], ["Power BI", "Statistics"]],
        "Credit Analyst": [["Financial Analysis", "Credit Risk"], ["Underwriting", "Excel"], ["Loan Review", "Risk Assessment"]],
        "Loan Officer": [["Customer Service", "Loan Origination"], ["Credit Reports", "Regulatory Knowledge"], ["Loan Processing", "CRM"]],
        "Compliance Analyst": [["Regulatory Compliance", "AML"], ["KYC", "Audit"], ["Policy Review", "Banking Laws"]],
        "Branch Manager": [["Leadership", "Sales"], ["Branch Operations", "Customer Experience"], ["Staff Training", "Team Management"]],
        "Bank Teller": [["Cash Handling", "Customer Service"], ["POS Systems", "Transaction Processing"], ["Fraud Detection", "Bank Procedures"]],
        "Risk Manager": [["Risk Modeling", "Market Risk"], ["Operational Risk", "Stress Testing"], ["Scenario Analysis", "Compliance"]],
        "Investment Analyst": [["Equity Research", "Valuation"], ["Portfolio Management", "Bloomberg"], ["Risk-Return Analysis", "Financial Modeling"]],
        "BI Analyst": [["Power BI", "Data Visualization"], ["Excel", "SQL"], ["Tableau", "Business Intelligence"]],
        "IT Auditor": [["Internal Controls", "IT Security"], ["Audit Tools", "Compliance Testing"], ["Access Management", "Risk Auditing"]],
        "Fraud Investigator": [["Transaction Monitoring", "Fraud Detection"], ["Forensics", "Surveillance"], ["Case Review", "Reporting"]],
        "Financial Advisor": [["Investment Strategy", "Client Communication"], ["Retirement Planning", "Wealth Management"], ["Portfolio Review", "Risk Tolerance"]],
        "Treasury Analyst": [["Cash Management", "Liquidity Planning"], ["Bank Transfers", "Forecasting"], ["Treasury Operations", "Funding"]],
        "Operations Analyst": [["Process Improvement", "Workflow Analysis"], ["Data Entry", "Reporting"], ["Bank Ops", "Compliance Checks"]],
        "Mortgage Underwriter": [["Credit Reports", "Loan-to-Value"], ["Income Verification", "DTI"], ["Appraisals", "Underwriting Guidelines"]],
        "Internal Auditor": [["Internal Controls", "Risk Assessment"], ["Audit Planning", "Compliance Testing"], ["Documentation", "Reporting"]],
        "AML Analyst": [["Anti-Money Laundering", "SAR Filing"], ["KYC Review", "Transaction Patterns"], ["Customer Due Diligence", "Risk Scoring"]],
        "Product Manager": [["Market Research", "Product Development"], ["Roadmapping", "Stakeholder Communication"], ["KPIs", "Customer Feedback"]],
        "Customer Success Manager": [["Client Retention", "Onboarding"], ["CRM", "Upselling"], ["Customer Satisfaction", "Product Adoption"]],
        "Cybersecurity Analyst": [["Intrusion Detection", "SIEM"], ["Firewalls", "Incident Response"], ["Threat Intelligence", "Endpoint Security"]],
    }
    
    expanded_data = []  # List to hold expanded employee data
    id_counter = 1  # Initialize employee ID counter

    # Loop through each job title in the defined skills mapping
    for job, skill_sets in bank_jobs_skills.items():
        for i in range(20):  # Generate 20 samples for each job title
            skill_set = random.choice(skill_sets)  # Randomly choose a skill set for the employee
            expanded_data.append({
                "employee_id": id_counter,  # Unique ID for the employee
                "skills": skill_set,        # Skills chosen from the random skill set
                "job": job                  # Job title associated with the employee
            })
            id_counter += 1  # Increment ID counter for the next employee

    return pd.DataFrame(expanded_data)  # Return the DataFrame containing employee records

def preprocess_data(df):
    """Prepares data for model training and evaluation.
    
    This function takes a DataFrame with employee skills and job titles, encodes the skills
    using One-Hot encoding, and factorizes job titles into numeric labels.
    The data is split into training and testing sets, with remapping of job labels for compatibility 
    during model training.
    
    Args:
        df (DataFrame): The DataFrame containing employee data with skills and job titles.
    
    Returns:
        tuple: Contains the training features (X_train), testing features (X_test),
               original training labels (y_train_orig), testing labels (y_test),
               the job classes, and the reverse mapping of labels.
    """
    mlb = MultiLabelBinarizer()  # Initialize MultiLabelBinarizer for skill encoding
    X = mlb.fit_transform(df['skills'])  # Encode skills into a binary format
    
    # Factorize job titles into numeric labels
    df['job_label'], job_classes = pd.factorize(df['job'])
    y = df['job_label']  # Store the numeric labels for jobs

    # Split the dataset into training and testing subsets (70% train, 30% test)
    X_train, X_test, y_train_orig, y_test_orig = train_test_split(X, y, test_size=0.3, random_state=42)

    # Create a mapping of original job labels to new indices for compatibility
    unique_train_labels = np.unique(y_train_orig)  # Get unique job labels from training set
    label_map = {old: new for new, old in enumerate(unique_train_labels)}  # New label mapping
    reverse_map = {v: k for k, v in label_map.items()}  # Reverse mapping for decoding predictions

    # Filter out invalid labels in the test set to maintain compatibility with training set
    valid_test_mask = [label in label_map for label in y_test_orig]
    X_test = X_test[valid_test_mask]  # Filter valid test features
    y_test_orig = y_test_orig[valid_test_mask]  # Filter valid test labels
    y_test = np.array([label_map[label] for label in y_test_orig])  # Remap valid test labels

    return X_train, X_test, y_train_orig, y_test, job_classes, reverse_map  # Return processed datasets and mappings

def train_model(X_train, y_train):
    """Trains a OneVsRestClassifier using Logistic Regression.
    
    This function fits a logistic regression model wrapped in a One-vs-Rest classifier 
    on the training data provided.
    
    Args:
        X_train (array): The training feature set (skills).
        y_train (array): The training labels (job titles).
    
    Returns:
        model: The trained OneVsRest model.
    """
    model = OneVsRestClassifier(LogisticRegression(max_iter=2000))  # Initialize model
    model.fit(X_train, y_train)  # Fit model to training data
    return model  # Return the trained model

def evaluate_model(model, X_test, y_test, job_classes, reverse_map):
    """Evaluates the model and prints a classification report.
    
    Using the trained model, this function makes predictions on the test set,
    generates a classification report to assess model performance, and prints results 
    including precision, recall, and F1-score for each job title.
    
    Args:
        model: The trained model to evaluate.
        X_test (array): The testing feature set (skills).
        y_test (array): The true labels for testing data (job titles).
        job_classes (array): The array of original job titles.
        reverse_map (dict): The mapping of label indices back to original job titles.
    """
    if len(y_test) > 0:  # Ensure there are labels to evaluate
        y_pred = model.predict(X_test)  # Make predictions on the test set
        test_labels_present = sorted(np.unique(y_test))  # Get unique labels present in test set
        # Map predicted label indices back to original job titles
        mapped_classes = [job_classes[reverse_map[i]] for i in test_labels_present]

        # Generate classification report providing metrics for each class
        report = classification_report(
            y_test, y_pred,
            labels=test_labels_present,
            target_names=mapped_classes,
            zero_division=0  # Prevent division by zero in metrics
        )
        print("\nClassification Report:\n")
        print(report)  # Print the classification report
    else:
        print("\nWarning: No valid job labels in the test set after mapping.\n")  # Alert if no valid labels

if __name__ == "__main__":
    # Generate and preprocess data
    df = generate_dummy_data()  # Create a dummy dataset of bank jobs and skills
    X_train, X_test, y_train, y_test, job_classes, reverse_map = preprocess_data(df)  # Process data for training

    # Train the model
    model = train_model(X_train, y_train)  # Fit the model to the training data

    # Evaluate the model
    evaluate_model(model, X_test, y_test, job_classes, reverse_map)  # Assess model performance

    # View sample data
    df['job_encoded'] = y_train  # Add the encoded job labels to the DataFrame for reference
    print("\nSample Data:\n")
    print(df.head(50))  # Display the first 50 entries of the DataFrame with employee IDs, skills, jobs, and encoded job labels



Classification Report:

                          precision    recall  f1-score   support

            Data Analyst       1.00      0.89      0.94         9
          Credit Analyst       1.00      1.00      1.00         6
            Loan Officer       1.00      1.00      1.00         6
      Compliance Analyst       1.00      1.00      1.00        11
          Branch Manager       1.00      1.00      1.00         5
             Bank Teller       1.00      1.00      1.00         8
            Risk Manager       1.00      1.00      1.00         5
      Investment Analyst       1.00      1.00      1.00         6
              BI Analyst       0.75      1.00      0.86         3
              IT Auditor       1.00      1.00      1.00         3
      Fraud Investigator       1.00      1.00      1.00         3
       Financial Advisor       1.00      1.00      1.00         8
        Treasury Analyst       1.00      1.00      1.00         5
      Operations Analyst       1.00      1.00     